In [ ]:
import numpy as np

data = np.load('/Users/romanos/Documents/BeatBot/data/BeatThis/ballroom.npz')

print(data.files)

# Each track has a mel spectrogram  
spec = data['ballroom_Albums-AnaBelen_Veneo-01/track_ts20']  # shape: (num_frames, 128)

print(spec.shape)
print(spec)

['ballroom_Albums-AnaBelen_Veneo-01/track', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps-5', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps-4', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps-3', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps-2', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps-1', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps1', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps2', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps3', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps4', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps5', 'ballroom_Albums-AnaBelen_Veneo-01/track_ps6', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts-20', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts-16', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts-12', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts-8', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts-4', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts4', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts8', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts12', 'ballroom_Albums-AnaBelen_Veneo-01/track_ts16', '

The annotations are stored as text files with beat and downbeat timestamps.
​

Structure:
text
```
beat_this_annotations/
├── gtzan/
│   ├── blues.00000.beats
│   ├── blues.00001.beats
│   ├── electronic.00024.beats
│   └── ...
├── ballroom/
│   ├── Albums-A_Tantas_Cosas-01.beats
│   ├── Albums-A_Tantas_Cosas-02.beats
│   └── ...
└── README.md
File Format (.beats):
Each .beats file is a plain text file with one beat per line:
```
```
text
time_seconds   beat_number
0.465          1
0.931          2
1.397          3
1.863          4
2.329          1   <- downbeat (new bar)
2.795          2
```

In [ ]:
import numpy as np

def load_beat_annotations(filepath):
    """Load beat annotations from a .beats file."""
    data = np.loadtxt(filepath)
    beat_times = data[:, 0]          # All beat times
    beat_positions = data[:, 1].astype(int)  # Beat positions (1,2,3,4)
    
    # Extract downbeats (beat_position == 1)
    downbeat_mask = beat_positions == 1
    downbeat_times = beat_times[downbeat_mask]
    
    return beat_times, beat_positions, downbeat_times


FEATURES EXTRACTED FROM ANNOTATIONS

In [ ]:
beats, positions, downbeats = load_beat_annotations(ann_file)
# beats: all beat times [0.465, 0.931, 1.397, ...]
# downbeats: bar starts [0.465, 2.329, 4.193, ...]

# Calculate from beat intervals
beat_intervals = np.diff(beats)
avg_interval = np.median(beat_intervals)
tempo = 60.0 / avg_interval  # BPM

# Every 4 beats = 1 bar (assuming 4/4 time)
# Every 8 beats = 1 phrase (your `bars` field)
bars_4beat = downbeats
bars_8beat = beats[::8]  # Every 8 beats

# Inferred from beat positions
beats_per_bar = positions.max()  # Usually 4 for 4/4 time



FEATURES EXTRACTED FROM SPECTROGRAMS

In [ ]:
def extract_energy_per_bar(spectrogram, bar_times, fps=50):
    """
    spectrogram: (128, T) mel spectrogram
    bar_times: times of bar boundaries (seconds)
    fps: frames per second (50 for Beat This)
    """
    energy_per_bar = []
    for i in range(len(bar_times) - 1):
        start_frame = int(bar_times[i] * fps)
        end_frame = int(bar_times[i+1] * fps)
        
        # Energy = sum of all mel bands in this time window
        bar_spec = spectrogram[:, start_frame:end_frame]
        energy = float(np.mean(bar_spec))  # Or np.sum, np.max
        energy_per_bar.append(energy)
    
    return np.array(energy_per_bar)

def extract_band_energies(spectrogram, bar_times, fps=50):
    """Extract low/mid/high frequency energy per bar."""
    # Mel bands roughly map to frequencies
    # For 128 bands from 30Hz to 11kHz:
    low_bands = spectrogram[:32, :]   # 30-250 Hz (bass/kick)
    mid_bands = spectrogram[32:96, :]  # 250-4000 Hz (vocals/melody)
    high_bands = spectrogram[96:, :]   # 4000-11000 Hz (hi-hats/cymbals)
    
    low_energy = []
    mid_energy = []
    high_energy = []
    
    for i in range(len(bar_times) - 1):
        start_frame = int(bar_times[i] * fps)
        end_frame = int(bar_times[i+1] * fps)
        
        low_energy.append(np.mean(low_bands[:, start_frame:end_frame]))
        mid_energy.append(np.mean(mid_bands[:, start_frame:end_frame]))
        high_energy.append(np.mean(high_bands[:, start_frame:end_frame]))
    
    return np.array(low_energy), np.array(mid_energy), np.array(high_energy)

def extract_spectral_flux(spectrogram, bar_times, fps=50):
    """Measure how much the spectrum changes (novelty)."""
    # Compute difference between consecutive frames
    flux = np.sum(np.abs(np.diff(spectrogram, axis=1)), axis=0)
    
    # Average flux per bar
    flux_per_bar = []
    for i in range(len(bar_times) - 1):
        start_frame = int(bar_times[i] * fps)
        end_frame = int(bar_times[i+1] * fps)
        flux_per_bar.append(np.mean(flux[start_frame:end_frame]))
    
    return np.array(flux_per_bar)


def detect_novelty_peaks(spectral_flux, threshold=1.5):
    """Find bars with high novelty (potential section changes)."""
    mean_flux = np.mean(spectral_flux)
    std_flux = np.std(spectral_flux)
    peaks = spectral_flux > (mean_flux + threshold * std_flux)
    return peaks  # Boolean array



❌ NOT Available (need raw audio or external tools):
These require original audio, which Beat This doesn't provide for most datasets:
​

❌ Harmonic key (need pitch class profile from audio)

❌ Vocal detection (need source separation or vocal model)

❌ Precise section labels (need manual annotation or advanced models)

❌ Camelot wheel code (depends on key detection)

For Beat This, leave them as None.

Creating an example track object

In [ ]:
import numpy as np
from pathlib import Path
from your_module import Track

def load_beatthis_track(dataset_name, track_id, 
                        npz_file, annotation_dir):
    """
    dataset_name: 'gtzan' or 'ballroom'
    track_id: e.g., 'electronic.00024'
    npz_file: path to gtzan.npz
    annotation_dir: path to beat_this_annotations/gtzan/
    """
    
    #Load spectrogram
    data = np.load(npz_file)
    spec = data[track_id]  # shape (128, T)
    duration = spec.shape[1] / 50.0  # 50 fps
    
    # Load annotations
    ann_file = Path(annotation_dir) / f"{track_id}.beats"
    ann_data = np.loadtxt(ann_file)
    beat_times = ann_data[:, 0]
    beat_positions = ann_data[:, 1].astype(int)
    downbeat_mask = beat_positions == 1
    downbeat_times = beat_times[downbeat_mask]
    
    # Compute tempo
    beat_intervals = np.diff(beat_times)
    tempo = 60.0 / np.median(beat_intervals)
    
    # Compute bars (every 8 beats = 2-bar phrase)
    bars = beat_times[::8]
    
    # Extract features from spectrogram
    low_energy, mid_energy, high_energy = extract_band_energies(
        spec, bars, fps=50
    )
    energy_per_bar = extract_energy_per_bar(spec, bars, fps=50)
    
    # Create Track object
    track = Track(
        track_id=f"{dataset_name}_{track_id}",
        source=f"beat_this_{dataset_name}",
        audio_path=None,  # No audio provided
        duration=duration,
        tempo=tempo,
        beats=beat_times,
        bars=bars,
        section_boundaries=np.array([]),  # Not available
        section_types=[],
        energy_per_bar=energy_per_bar,
        low_band_energy=low_energy,
        mid_band_energy=mid_energy,
        high_band_energy=high_energy,
        vocal_mask=None,  # Can't detect without audio
        key_tonic=None,
        key_scale=None,
        key_confidence=None,
        # No cue_in/cue_out labels (will use defaults)
    )
    
    return track

# Usage
track = load_beatthis_track(
    dataset_name='gtzan',
    track_id='electronic.00024',
    npz_file='gtzan.npz',
    annotation_dir='beat_this_annotations/gtzan'
)

print(f"Track: {track.track_id}")
print(f"Duration: {track.duration:.1f}s, Tempo: {track.tempo:.1f} BPM")
print(f"Beats: {len(track.beats)}, Bars: {len(track.bars)}")
print(f"Energy per bar: {track.energy_per_bar[:5]}")
